In [3]:
import pandas as pd
import numpy as np
import time
# import shap
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier

## loading data

In [4]:
CONS_PATH = "data/consdf.parquet"
ACCT_PATH = "data/acctdf.parquet"
TRXN_PATH = "data/trxndf.parquet"
CATMAP_PATH = "data/cat_map.csv"

In [5]:
# Load data
consdf = pd.read_parquet(CONS_PATH)
consdf_full = pd.read_parquet(CONS_PATH)
acctdf = pd.read_parquet(ACCT_PATH)
trxndf = pd.read_parquet(TRXN_PATH)
cat_map = pd.read_csv(CATMAP_PATH)

print("consdf:", consdf.shape)
print("acctdf:", acctdf.shape)
print("trxndf:", trxndf.shape)
print("cat_map:", cat_map.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/uss/hdsi-prismdata/q2-ucsd-consDF.pqt'

In [6]:
consdf

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
0,0,2021-09-01,726.0,0.0
1,1,2021-07-01,626.0,0.0
2,2,2021-05-01,680.0,0.0
3,3,2021-03-01,734.0,0.0
4,4,2021-10-01,676.0,0.0
...,...,...,...,...
14995,14995,2022-03-08,655.0,NaN
14996,14996,2022-01-15,625.0,NaN
14997,14997,2022-01-31,688.0,NaN
14998,14998,2022-03-08,722.0,NaN


In [7]:
acctdf

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,3023,0,SAVINGS,2021-08-31,90.57
1,3023,1,CHECKING,2021-08-31,225.95
2,4416,2,SAVINGS,2022-03-31,15157.17
3,4416,3,CHECKING,2022-03-31,66.42
4,4227,4,CHECKING,2021-07-31,7042.90
...,...,...,...,...,...
24461,11500,24461,CHECKING,2022-03-27,732.75
24462,11615,24462,SAVINGS,2022-03-30,5.00
24463,11615,24463,CHECKING,2022-03-30,1956.46
24464,12210,24464,CHECKING,2022-03-28,2701.51


In [8]:
trxndf

,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date
0,3023,0,4,0.05,CREDIT,2021-04-16
1,3023,1,12,481.56,CREDIT,2021-04-30
2,3023,2,4,0.05,CREDIT,2021-05-16
3,3023,3,4,0.07,CREDIT,2021-06-16
4,3023,4,4,0.06,CREDIT,2021-07-16
...,...,...,...,...,...,...
6407316,10533,6405304,31,4.96,DEBIT,2022-03-11
6407317,10533,6405305,12,63.48,DEBIT,2022-03-30
6407318,10533,6405306,12,53.99,DEBIT,2022-03-30
6407319,10533,6405307,12,175.98,DEBIT,2022-03-31


## data cleaning/prepping

In [9]:
consdf = consdf.copy()
consdf["evaluation_date"] = pd.to_datetime(consdf["evaluation_date"], errors="coerce")

# # drop missing DQ_TARGET
# consdf = consdf[consdf["DQ_TARGET"].notna()].copy()
# consdf["DQ_TARGET"] = consdf["DQ_TARGET"].astype(int)

acctdf = acctdf.copy()
acctdf["balance_date"] = pd.to_datetime(acctdf["balance_date"], errors="coerce")

trxndf = trxndf.copy()
trxndf["posted_date"] = pd.to_datetime(trxndf["posted_date"], errors="coerce")

# Deduplicate transactions (use this whenever you build transaction features)
trxndf = (
    trxndf.sort_values(["posted_date"])
      .drop_duplicates(subset=["prism_transaction_id"], keep="first")
)


## scoring exclusions

In [10]:
# Accounts: how many accounts + how many balance snapshots
acct_stats = (
    acctdf.groupby("prism_consumer_id")
    .agg(
        n_accounts=("prism_account_id", "nunique"),
        n_balance_days=("balance_date", "nunique"),
        first_balance=("balance_date", "min"),
        last_balance=("balance_date", "max"),
    )
    .reset_index()
)

# Transactions: count + span + credits
tx_stats = (
    trxndf.groupby("prism_consumer_id")
    .agg(
        n_txn=("prism_transaction_id", "count"),
        first_txn=("posted_date", "min"),
        last_txn=("posted_date", "max"),
    )
    .reset_index()
)
tx_stats["txn_span_days"] = (tx_stats["last_txn"] - tx_stats["first_txn"]).dt.days

credit_stats = (
    trxndf.assign(is_credit=(trxndf["credit_or_debit"] == "CREDIT").astype(int))
    .groupby("prism_consumer_id")
    .agg(n_credit=("is_credit", "sum"))
    .reset_index()
)
credit_debit_counts = (
    trxndf.groupby(["prism_consumer_id", "credit_or_debit"])
          .size()
          .unstack(fill_value=0)   # creates CREDIT and DEBIT columns
          .reset_index()
)

# rename to match your naming convention
credit_debit_counts = credit_debit_counts.rename(columns={
    "CREDIT": "n_credit_txn",
    "DEBIT": "n_debit_txn"
})

# Combine into one scoring table (one row per consumer)
scoring = (
    consdf[["prism_consumer_id", "evaluation_date", "DQ_TARGET", "credit_score"]]
    .merge(acct_stats, on="prism_consumer_id", how="left")
    .merge(tx_stats, on="prism_consumer_id", how="left")
    .merge(credit_stats, on="prism_consumer_id", how="left")
    .merge(credit_debit_counts,on="prism_consumer_id",how="left")
)

# Fill missing stats with 0 where appropriate
for col in ["n_accounts", "n_balance_days", "n_txn", "txn_span_days", "n_credit"]:
    if col in scoring.columns:
        scoring[col] = scoring[col].fillna(0)


In [11]:
# consumers with no accounts
consumers_with_accounts = set(acctdf["prism_consumer_id"].unique())
all_consumers = set(consdf["prism_consumer_id"].unique())

no_account_ids = list(all_consumers - consumers_with_accounts)

# print("Total consumers:", len(all_consumers))
# print("Consumers with NO accounts:", len(no_account_ids))

Total consumers: 15000
Consumers with NO accounts: 1991


In [12]:
# checking to see how many "no accounts" have transactions
trx_no_account = trxndf[trxndf["prism_consumer_id"].isin(no_account_ids)]

consumers_no_account_with_txn = trx_no_account["prism_consumer_id"].nunique()

# print("\nConsumers with NO accounts but WITH transactions:", consumers_no_account_with_txn)


# # 3️⃣ Total transaction rows for these consumers
# print("Total transaction rows for these consumers:", trx_no_account.shape[0])


# # 4️⃣ Show example consumer IDs
# print("\nExample consumer IDs (no account but with transactions):")
# print(list(trx_no_account["prism_consumer_id"].unique())[:5])


# # 5️⃣ Show sample transaction rows
# print("\nSample transaction rows:")
# display(trx_no_account.head(10))


Consumers with NO accounts but WITH transactions: 1845
Total transaction rows for these consumers: 1204382

Example consumer IDs (no account but with transactions):
['1706', '3327', '2042', '1970', '114']

Sample transaction rows:


,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date
858236,1706,857660,0,2000.00,DEBIT,2020-11-01
895947,3327,895260,0,300.00,CREDIT,2020-11-01
895953,3327,895266,0,200.00,DEBIT,2020-11-02
895954,3327,895267,0,100.00,DEBIT,2020-11-06
728042,2042,727489,2,500.00,CREDIT,2020-11-12
858237,1706,857661,1,5000.00,DEBIT,2020-11-12
728043,2042,727490,4,25.00,CREDIT,2020-11-18
728044,2042,727491,2,500.00,CREDIT,2020-11-18
728045,2042,727492,4,475.00,CREDIT,2020-11-19
88530,1970,88466,4,0.05,CREDIT,2020-11-20


In [16]:
# scoring["n_txn"].describe()


count    15000.000000
mean       427.020600
std        397.534424
min          0.000000
25%        154.000000
50%        337.000000
75%        589.000000
max       8478.000000
Name: n_txn, dtype: float64

In [17]:
# scoring["txn_span_days"].describe()


count    15000.000000
mean       152.998600
std         78.290821
min          0.000000
25%         88.000000
50%        178.000000
75%        242.000000
max        801.000000
Name: txn_span_days, dtype: float64

even the low activity consumers have 88 days (2-3 months) of transaction history

In [18]:
# scoring["n_credit"].describe()

count    15000.000000
mean        73.636733
std         80.685765
min          0.000000
25%         28.000000
50%         52.000000
75%         93.000000
max       1553.000000
Name: n_credit, dtype: float64

bottom 25% has 28 credit transactions

In [19]:
RULES = {

    # No financial footprint at all
    "no_accounts": scoring["n_accounts"] < 1,

    # No transaction history
    "no_transactions": scoring["n_txn"] < 1,

    # Must have at least 1 credit and 1 debit
    "no_credit_txn": scoring["n_credit_txn"] < 1,
    "no_debit_txn": scoring["n_debit_txn"] < 1,

    # Too short of observable history
    "short_txn_history": scoring["txn_span_days"] < 30,
}

# Apply rules
for name, mask in RULES.items():
    scoring[name] = mask

# Exclusion flag
scoring["excluded"] = scoring[list(RULES.keys())].any(axis=1)

# Summary
# print("Total consumers:", scoring.shape[0])
# print("Excluded:", scoring["excluded"].sum())
# print("Eligible:", (~scoring["excluded"]).sum())
# print("Exclusion rate:", scoring["excluded"].mean())


Total consumers: 15000
Excluded: 3004
Eligible: 11996
Exclusion rate: 0.20026666666666668


In [20]:
eligible_ids = scoring.loc[~scoring["excluded"], "prism_consumer_id"]

# print("Eligible consumers:", len(eligible_ids))


Eligible consumers: 11996


In [21]:
consdf_eligible = consdf[
    consdf["prism_consumer_id"].isin(eligible_ids)
].copy()

acctdf_eligible = acctdf[
    acctdf["prism_consumer_id"].isin(eligible_ids)
].copy()

trxndf_eligible = trxndf[
    trxndf["prism_consumer_id"].isin(eligible_ids)
].copy()


In [22]:
# print("Consumers in consdf_eligible:", consdf_eligible["prism_consumer_id"].nunique())
# print("Consumers in acctdf_eligible:", acctdf_eligible["prism_consumer_id"].nunique())
# print("Consumers in trxndf_eligible:", trxndf_eligible["prism_consumer_id"].nunique())


Consumers in consdf_eligible: 11996
Consumers in acctdf_eligible: 11996
Consumers in trxndf_eligible: 11996


In [23]:
holdout_ids = consdf_full.loc[
    consdf_full["DQ_TARGET"].isna(),
    "prism_consumer_id"
]

In [24]:
scoring_holdout = scoring[
    scoring["prism_consumer_id"].isin(holdout_ids)
].copy()

In [25]:
# print("Holdout consumers:", scoring_holdout.shape[0])
# print("Excluded in holdout:", scoring_holdout["excluded"].sum())
# print("Eligible in holdout:", (~scoring_holdout["excluded"]).sum())
# print("Holdout exclusion rate:",
      round(scoring_holdout["excluded"].mean(), 4))

Holdout consumers: 3000
Excluded in holdout: 626
Eligible in holdout: 2374
Holdout exclusion rate: 0.2087


## feature engineering

In [26]:
initial_df = (
    acctdf
    .merge(consdf, on='prism_consumer_id', how='inner')
    .groupby(['prism_consumer_id'])
    .agg(
        balance=('balance', 'sum'),
        balance_date=('balance_date', 'max')
    )
    .reset_index()
).merge(trxndf,on='prism_consumer_id')

In [27]:
mapping = dict(zip(cat_map["category_id"], cat_map["category"]))
initial_df["category"] = initial_df["category"].replace(mapping)
monthly_summary=initial_df.copy()
monthly_summary['amount'] = np.where(initial_df['credit_or_debit'] == 'DEBIT', -initial_df['amount'],initial_df['amount'])
monthly_summary['posted_date'] = pd.to_datetime(monthly_summary['posted_date'])
monthly_summary = (
    monthly_summary
    .groupby(['prism_consumer_id', monthly_summary['posted_date'].dt.to_period('M')])
    .agg(
        starting_balance=('balance', 'first'),
        monthly_total=('balance', 'sum'),
        trxndf_count = ('balance', 'count')
    )
    .reset_index()
)
monthly_summary['posted_date'] = monthly_summary['posted_date'].dt.to_timestamp()

In [28]:
monthly_summary = monthly_summary.merge(consdf[['prism_consumer_id','DQ_TARGET']],on='prism_consumer_id').dropna()


In [29]:
# ensure date type
monthly_summary["posted_date"] = pd.to_datetime(monthly_summary["posted_date"])

# sort properly
monthly_summary = monthly_summary.sort_values(["prism_consumer_id", "posted_date"])

# calculate running balance
monthly_summary["monthly_balance"] = (
    monthly_summary["starting_balance"]
    + monthly_summary.groupby("prism_consumer_id")["monthly_total"].cumsum()
)

In [30]:
del_df = monthly_summary[monthly_summary['DQ_TARGET'] == 1]
nondel_df = monthly_summary[monthly_summary['DQ_TARGET'] == 0]
ids_1 = del_df["prism_consumer_id"].dropna().unique()
ids_0 = del_df["prism_consumer_id"].dropna().unique()

In [31]:
mtotal_df = monthly_summary.groupby('prism_consumer_id').agg(
        DQ_TARGET = ('DQ_TARGET', 'first'),
        monthly_mean=('monthly_total', 'mean'),
        monthly_max=('monthly_total', 'max'),
        monthly_min=('monthly_total', 'min'),
        trxndf_count = ('trxndf_count','first'),
        month_count=('monthly_total', 'count')
    )

In [32]:
cd_df = initial_df[['prism_consumer_id','amount','credit_or_debit']].groupby(['prism_consumer_id','credit_or_debit']).sum().reset_index()


In [33]:
cd_df = (
    cd_df
    .pivot_table(
        index='prism_consumer_id',
        columns='credit_or_debit',
        values='amount',
        aggfunc='sum',
        fill_value=0
    )
    .assign(
        credit_debit_ratio=lambda x: x['CREDIT'] / (x['DEBIT'] + 1),
        net_flow=lambda x: x['CREDIT'] - x['DEBIT']
    )
)

In [34]:
cd_df = cd_df.reset_index().merge(consdf[['prism_consumer_id','DQ_TARGET']],on='prism_consumer_id').dropna()


In [35]:
net_df = initial_df[['prism_consumer_id','posted_date','category','credit_or_debit','amount']].copy()
net_df['amount'] = np.where(net_df['credit_or_debit'] == 'DEBIT', -net_df['amount'],net_df['amount'])
net_df['posted_date'] = pd.to_datetime(net_df['posted_date'])
net_df['month'] = net_df['posted_date'].dt.to_period('M')
mn_df = net_df.groupby(['prism_consumer_id','month']).agg(
        monthly_total=('amount', 'sum'),
        monthly_std =('amount','std')
    ).reset_index()


monthly features

In [36]:
monthly_features = mn_df.groupby(['prism_consumer_id']).agg(
    monthly_net_total=('monthly_total', 'sum'),
    monthly_net_avg=('monthly_total', 'mean'),
    monthly_net_max=('monthly_total', 'max'),
    monthly_net_min=('monthly_total', 'min'),
    monthly_std_avg=('monthly_std', 'mean')
).reset_index().merge(consdf[['prism_consumer_id','DQ_TARGET']],on='prism_consumer_id').dropna()
monthly_features['prism_consumer_id'] = monthly_features['prism_consumer_id'].astype(int)
mtotal_df = mtotal_df.reset_index()
mtotal_df['prism_consumer_id'] = mtotal_df['prism_consumer_id'].astype(int)
cd_df['prism_consumer_id'] = cd_df['prism_consumer_id'].astype(int)
monthly_features['net_range'] = monthly_features['monthly_net_max'] - monthly_features['monthly_net_min']

In [37]:
initial_df['amount'] = np.where(initial_df['credit_or_debit'] == 'DEBIT', -initial_df['amount'],initial_df['amount'])
cat_df = initial_df.groupby(['prism_consumer_id','category'])['amount'].sum().reset_index()

In [38]:
cat_pivot = (
    cat_df
    .pivot(
        index='prism_consumer_id',
        columns='category',
        values='amount'
    )
    .fillna(0)
)

In [39]:
outflows = cat_pivot.clip(upper=0).abs()
inflows  = cat_pivot.clip(lower=0)

cat_features = pd.DataFrame(index=cat_pivot.index)

cat_features['total_outflows'] = outflows.sum(axis=1)
cat_features['total_inflows']  = inflows.sum(axis=1)
cat_features['net_flow']       = cat_pivot.sum(axis=1)

In [40]:
for col in outflows.columns:
    cat_features[f'{col}_outflow_ratio'] = (
        outflows[col] / (cat_features['total_outflows'] + 1))

In [41]:
# Income reliance
cat_features['paycheck_ratio'] = (
    inflows.get('PAYCHECK', 0) / (cat_features['total_inflows'] + 1)
)

# Cash usage
cat_features['atm_cash_ratio'] = (
    outflows.get('ATM_CASH', 0) / (cat_features['total_outflows'] + 1)
)

# Entertainment vs essentials proxy
cat_features['entertainment_ratio'] = (
    outflows.get('ENTERTAINMENT', 0) / (cat_features['total_outflows'] + 1)
)

# Refund dependence
cat_features['refund_ratio'] = (
    inflows.get('REFUND', 0) / (cat_features['total_inflows'] + 1)
)

In [42]:
outflows = outflows.reset_index().merge(consdf[['prism_consumer_id','DQ_TARGET']],on='prism_consumer_id').dropna()


In [43]:
cat_features = cat_features.reset_index().merge(consdf[['prism_consumer_id','DQ_TARGET']],on='prism_consumer_id').dropna()


In [44]:
add_df = cat_features[['prism_consumer_id','refund_ratio','paycheck_ratio']].copy()
add_df['prism_consumer_id'] = add_df['prism_consumer_id'].astype(int)
outflows['prism_consumer_id'] = outflows['prism_consumer_id'].astype(int)
out_df = outflows.copy()

In [45]:
initial_df['amount'] = np.where(initial_df['credit_or_debit'] == 'DEBIT', -initial_df['amount'],initial_df['amount'])
cat_df = initial_df.groupby(['prism_consumer_id','category'])['amount'].mean().reset_index()

In [46]:
cat_pivot = (
    cat_df
    .pivot(
        index='prism_consumer_id',
        columns='category',
        values='amount'
    )
    .fillna(0)
)
cat_pivot.columns = cat_pivot.columns + "_trxnavg"
cat_pivot = cat_pivot.reset_index().merge(consdf[['prism_consumer_id','DQ_TARGET']],on='prism_consumer_id').dropna()
cat_pivot['prism_consumer_id'] = cat_pivot['prism_consumer_id'].astype(int)

income

In [47]:
mapping = dict(zip(cat_map["category_id"], cat_map["category"]))
trxndf["category"] = trxndf["category"].replace(mapping)

income_categories = [
    'PAYCHECK',
    'DEPOSIT',
    'UNEMPLOYMENT_BENEFITS',
    'OTHER_BENEFITS',
    'PENSION',
    'INVESTMENT_INCOME'
]

income_df = trxndf[
    trxndf['category'].isin(income_categories)
].copy()
income_df['prism_transaction_id'].duplicated().sum()
income_df['posted_date'] = pd.to_datetime(income_df['posted_date'])

In [48]:
income_time = (
    income_df
    .groupby('prism_consumer_id')
    .agg(
        first_income_date=('posted_date', 'min'),
        last_income_date=('posted_date', 'max')
    )
    .reset_index()
)

income_time['income_span_days'] = (
    income_time['last_income_date'] - income_time['first_income_date']
).dt.days

In [49]:
income_df = income_time[['prism_consumer_id','income_span_days']]
income_df['prism_consumer_id'] = income_time['prism_consumer_id'].astype(int)

/tmp/ipykernel_3038/1562562249.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  income_df['prism_consumer_id'] = income_time['prism_consumer_id'].astype(int)


preliminary testing

In [50]:
cat_pivot= cat_pivot.drop(columns='DQ_TARGET')


In [51]:
main_df= monthly_features.merge(mtotal_df,on='prism_consumer_id')
main_df['DQ_TARGET'] = main_df['DQ_TARGET_x']
main_df = main_df.drop(columns=['DQ_TARGET_x','DQ_TARGET_y'])
cd_df = cd_df.drop(columns=['net_flow','DQ_TARGET'])
main_df= main_df.merge(cd_df,on='prism_consumer_id')
main_df= main_df.merge(add_df,on='prism_consumer_id')
main_df= main_df.merge(out_df,on='prism_consumer_id')
main_df= main_df.merge(income_df,on='prism_consumer_id')
main_df= main_df.merge(cat_pivot,on='prism_consumer_id')
main_df

,prism_consumer_id,monthly_net_total,monthly_net_avg,monthly_net_max,monthly_net_min,monthly_std_avg,net_range,monthly_mean,monthly_max,monthly_min,...,REFUND_trxnavg,RENT_trxnavg,RISK_CATCH_ALL_trxnavg,RTO_LTO_trxnavg,SELF_TRANSFER_trxnavg,TAX_trxnavg,TIME_OR_STUFF_trxnavg,TRANSPORATION_trxnavg,TRAVEL_trxnavg,UNEMPLOYMENT_BENEFITS_trxnavg
0,0,-521.59,-74.512857,830.73,-2584.24,213.544425,3414.97,1.867299e+04,27231.45,8970.36,...,19.960000,0.000000,0.0,0.0,116.685652,867.840,0.000000,2.480000,54.375000,0.0
1,1,1805.43,257.918571,1109.02,-940.73,292.763392,2049.75,1.481371e+05,208052.46,102375.02,...,2.420000,0.000000,0.0,0.0,233.410256,1162.700,0.000000,25.900000,0.000000,0.0
2,10,-1190.04,-170.005714,431.40,-971.45,260.603079,1402.85,4.015226e+04,60169.52,19781.76,...,18.466000,103.000000,0.0,0.0,237.568750,0.000,0.000000,17.520000,0.000000,0.0
3,100,-4505.77,-750.961667,1276.72,-3332.81,832.186871,4609.53,5.399456e+04,63731.28,45142.99,...,1.468750,0.000000,0.0,0.0,547.296667,0.000,0.000000,0.000000,0.000000,0.0
4,1000,438.08,62.582857,2982.67,-2884.56,1223.790895,5867.23,2.871107e+03,3524.25,476.25,...,1.370000,0.000000,0.0,0.0,828.920370,0.000,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9589,995,21842.68,3120.382857,10212.74,-3244.20,1016.422674,13456.94,2.053630e+06,3162590.31,670852.49,...,1.463333,1134.158000,0.0,0.0,1250.937000,626.432,0.000000,0.000000,0.000000,0.0
9590,996,26713.18,3816.168571,41464.50,-16811.41,2623.765971,58275.91,0.000000e+00,0.00,0.00,...,2.944444,0.000000,0.0,0.0,3532.145542,0.000,0.000000,12.000000,493.909091,0.0
9591,997,-14899.66,-2128.522857,206.99,-3741.45,745.079512,3948.44,4.787867e+06,6756531.35,1404823.35,...,14.420000,0.000000,0.0,0.0,940.464154,2516.000,20.149683,0.000000,0.000000,0.0
9592,998,5507.73,786.818571,3359.83,-1022.35,537.836676,4382.18,7.987620e+05,1116887.94,275774.80,...,9.546429,231.316667,0.0,0.0,759.527885,2991.340,0.000000,12.700000,248.140000,0.0


In [52]:
# columns I will need: credit/debit, amount, posted date, evaluation date, prism consumer id, DQ_TARGET
merged = pd.merge(consdf.dropna(), trxndf, on='prism_consumer_id', how='left')

In [53]:
merged = merged[merged['posted_date'] <= merged['evaluation_date']]
credit_only = merged[merged['credit_or_debit'] == 'CREDIT'].copy()
credit_only['posted_date'] = pd.to_datetime(credit_only['posted_date'])
credit_only['Year-Month'] = credit_only['posted_date'].dt.to_period('M')
debt_only = trxndf[trxndf['credit_or_debit']=='DEBIT']
monthly_inflow = credit_only.groupby(['prism_consumer_id', 'Year-Month'])['amount'].sum().reset_index(name='monthly_inflow')
consdf['Evaluation Month'] = consdf['evaluation_date'].dt.to_period('M')
with_eval_month = pd.merge(consdf, monthly_inflow, on='prism_consumer_id', how='left')

In [54]:
with_eval_month['months_diff'] = (
    (with_eval_month['Evaluation Month'].dt.year - with_eval_month['Year-Month'].dt.year) * 12 +
    (with_eval_month['Evaluation Month'].dt.month - with_eval_month['Year-Month'].dt.month)
)
last_year = with_eval_month[(with_eval_month['months_diff'] >= 1) & (with_eval_month['months_diff'] <= 12)]
sum_yearly_inflow = last_year.groupby('prism_consumer_id')['monthly_inflow'].sum().reset_index(name='avg_yearly_inflow')
year_std = last_year.groupby('prism_consumer_id')['monthly_inflow'].std().reset_index()
year_std.columns = ['prism_consumer_id', 'std_inflow']

In [55]:
# Trend: Is income increasing or decreasing?
def calculate_trend(group):
    if len(group) < 2:
        return 0
    months = group['months_diff'].values
    inflows = group['monthly_inflow'].values
    return np.polyfit(months, inflows, 1)[0]  # slope

trend = last_year.groupby('prism_consumer_id').apply(calculate_trend, include_groups=False).reset_index()
trend.columns = ['prism_consumer_id', 'trend']
num_transactions = last_year.groupby('prism_consumer_id').size().reset_index()
num_transactions.columns = ['prism_consumer_id', 'num_transactions']

In [56]:
debt_only = trxndf[trxndf['credit_or_debit'] == 'DEBIT'].copy()
debt_only['posted_date'] = pd.to_datetime(debt_only['posted_date'])
# debt_only['category'] = debt_only['category'].astype(int)

# debt_with_category = pd.merge(debt_only, cat_map, left_on='category', right_on='category_id', how='left')[['prism_consumer_id',\
#     'prism_transaction_id', 'amount', 'credit_or_debit', 'posted_date', 'category_id', 'category_y']]
debt_with_category = debt_only.rename(columns={'category_y':'category'})
groceries_only = debt_with_category[debt_with_category['category']=='GROCERIES']

debt_with_eval = pd.merge(groceries_only, consdf[['prism_consumer_id', 'evaluation_date']], on='prism_consumer_id', how='left')

# Filter for transactions in the 3 months before evaluation_date
debt_with_eval['months_before_eval'] = (
    (debt_with_eval['evaluation_date'].dt.year - debt_with_eval['posted_date'].dt.year) * 12 +
    (debt_with_eval['evaluation_date'].dt.month - debt_with_eval['posted_date'].dt.month)
)

debt_9m = debt_with_eval[(debt_with_eval['months_before_eval'] >= 0) & 
                          (debt_with_eval['months_before_eval'] < 9)]

# total spend of groceries per consumer over a 9 month window (last 9 months before eval date)
total_spend_groceries_9m = debt_9m.groupby('prism_consumer_id')['amount'].sum().reset_index()
total_spend_groceries_9m.columns = ['prism_consumer_id', 'sum_groceries_9m']

In [57]:
# total spend of dining per consumer over a month window (last month before eval date)
dining_only = debt_with_category[debt_with_category['category']=='FOOD_AND_BEVERAGES']

debt_with_eval_dining = pd.merge(dining_only, consdf[['prism_consumer_id', 'evaluation_date']], on='prism_consumer_id', how='left')

# Filter for transactions in the 6 months before evaluation_date
debt_with_eval_dining['months_before_eval'] = (
    (debt_with_eval_dining['evaluation_date'].dt.year - debt_with_eval_dining['posted_date'].dt.year) * 12 +
    (debt_with_eval_dining['evaluation_date'].dt.month - debt_with_eval_dining['posted_date'].dt.month)
)

debt_6m = debt_with_eval_dining[(debt_with_eval_dining['months_before_eval'] >= 0) & 
                          (debt_with_eval_dining['months_before_eval'] < 6)]

# total spend of groceries per consumer over a 6 month window (last 6 months before eval date)
total_spend_dining_6m = debt_6m.groupby('prism_consumer_id')['amount'].sum().reset_index()
total_spend_dining_6m.columns = ['prism_consumer_id', 'sum_dining_6m']

In [58]:
# merge evaluation date ONCE
tx = debt_with_category.merge(
    consdf[['prism_consumer_id', 'evaluation_date']],
    on='prism_consumer_id',
    how='left'
)

tx = tx[tx['credit_or_debit'] == 'DEBIT']
tx['amount'] = tx['amount'].abs()

# numerator
total_spend_gambling = tx[tx['category'] == 'GAMBLING'].groupby('prism_consumer_id')['amount'].sum()

# denominator
total_spend_all = tx.groupby('prism_consumer_id')['amount'].sum()

pct_spend_gambling = (total_spend_gambling / total_spend_all).fillna(0).reset_index(name='pct_spend_gambling')

In [59]:
essentials = ['RENT', 'MORTGAGE', 'BILLS_UTILITIES', 'ESSENTIAL_SERVICES', 'GROCERIES', 'AUTOMOTIVE', 'TRANSPORTATION', \
'HEALTHCARE_MEDICAL', 'INSURANCE', 'CHILD_DEPENDENTS', 'PETS', 'TAX', 'LOAN', 'AUTO_LOAN', 'DEBT', 'CREDIT_CARD_PAYMENT', \
'EDUCATION', 'LEGAL', 'GOVERNMENT_SERVICES']

total_spend_essentials = tx[tx['category'].isin(essentials)].groupby('prism_consumer_id')['amount'].sum()

pct_spend_essentials = (total_spend_essentials / total_spend_all).reset_index()

pct_spend_essentials = pct_spend_essentials.rename(columns={'amount':'pct_spend_essentials'})

In [60]:
# # change in groceries per consumer from the 3 most recent months to the prior 3-6 months before evaluation date
# lowers AUC from 0.721 to 0.71

# recent 3 months (0–2)
recent_3m = debt_with_eval[(debt_with_eval['months_before_eval'] >= 0) & (debt_with_eval['months_before_eval'] < 3)]

recent_spend = recent_3m.groupby('prism_consumer_id')['amount'].sum().reset_index(name='groceries_0_3m')

# prior 3 months (3–5)
prior_3m = debt_with_eval[(debt_with_eval['months_before_eval'] >= 3) & (debt_with_eval['months_before_eval'] < 6)]

prior_spend = prior_3m.groupby('prism_consumer_id')['amount'].sum().reset_index(name='groceries_3_6m')

# merge and compute delta
delta_groceries_3m = recent_spend.merge(
    prior_spend,
    on='prism_consumer_id',
    how='outer'
).fillna(0)

delta_groceries_3m['delta_groceries_3m'] = delta_groceries_3m['groceries_0_3m'] - delta_groceries_3m['groceries_3_6m']

delta_groceries_3m = delta_groceries_3m[['prism_consumer_id', 'delta_groceries_3m']]

utilities = ['BILLS_UTILITIES', 'ESSENTIAL_SERVICES']

total_spend_utilities = tx[tx['category'].isin(utilities)].groupby('prism_consumer_id')['amount'].sum()

pct_spend_utilities = (total_spend_utilities / total_spend_all).reset_index()

pct_spend_utilities = pct_spend_utilities.rename(columns={'amount':'pct_spend_utilities'})

In [61]:
# has overdraft - 6 months
# Merge evaluation dates with ALL debt transactions
debt_with_eval = pd.merge(
    debt_with_category, 
    consdf[['prism_consumer_id', 'evaluation_date']], 
    on='prism_consumer_id', 
    how='left'
)

# Calculate days before evaluation
debt_with_eval['days_before_eval'] = (
    debt_with_eval['evaluation_date'] - debt_with_eval['posted_date']
).dt.days

# Filter for OVERDRAFT category AND within 6 months
overdraft_6m = debt_with_eval[
    (debt_with_eval['category'] == 'OVERDRAFT') &
    (debt_with_eval['days_before_eval'] >= 0) & 
    (debt_with_eval['days_before_eval'] <= 180)
]

# Group to get consumers with overdrafts
has_overdraft_6m = overdraft_6m.groupby('prism_consumer_id').size().reset_index(name='overdraft_count')
has_overdraft_6m['has_overdraft_6m'] = 1

has_overdraft_6m = has_overdraft_6m[['prism_consumer_id', 'has_overdraft_6m']]

In [62]:
# has account fees - 6 months
# Merge evaluation dates with ALL debt transactions
debt_with_eval = pd.merge(
    debt_with_category, 
    consdf[['prism_consumer_id', 'evaluation_date']], 
    on='prism_consumer_id', 
    how='left'
)

# Calculate days before evaluation
debt_with_eval['days_before_eval'] = (
    debt_with_eval['evaluation_date'] - debt_with_eval['posted_date']
).dt.days

# Filter for ACCOUNT FEES category AND within 6 months
acct_fees_6m = debt_with_eval[
    (debt_with_eval['category'] == 'ACCOUNT_FEES') &
    (debt_with_eval['days_before_eval'] >= 0) & 
    (debt_with_eval['days_before_eval'] <= 180)
]

# Group to get consumers with acct fee
has_acct_fee_6m = acct_fees_6m.groupby('prism_consumer_id').size().reset_index(name='acct_fees_count')
has_acct_fee_6m['has_acct_fee_6m'] = 1

has_acct_fee_6m = has_acct_fee_6m[['prism_consumer_id', 'has_acct_fee_6m']]

In [63]:
#atm cash ratio per consumer

debt_with_eval = pd.merge(
    debt_with_category,
    consdf[['prism_consumer_id', 'evaluation_date']],
    on='prism_consumer_id',
    how='left'
)

debt_with_eval['posted_date'] = pd.to_datetime(debt_with_eval['posted_date'])
debt_with_eval['evaluation_date'] = pd.to_datetime(debt_with_eval['evaluation_date'])

debt_with_eval = debt_with_eval[
    debt_with_eval['posted_date'] <= debt_with_eval['evaluation_date']
]

total_debt_spend = debt_with_eval.groupby('prism_consumer_id')['amount'].sum().reset_index(name='total_debit_spend')

In [64]:
atm_cash_spend = (
    debt_with_eval[debt_with_eval['category'] == 'ATM_CASH']
    .groupby('prism_consumer_id')['amount']
    .sum()
    .reset_index(name='atm_cash_spend')
)

atm_cash_ratio = total_debt_spend.merge(atm_cash_spend, on='prism_consumer_id',how='left').fillna(0)
atm_cash_ratio['atm_cash_ratio'] = atm_cash_ratio['atm_cash_spend'] / atm_cash_ratio['total_debit_spend']
atm_cash_ratio['atm_cash_ratio'] = (
    atm_cash_ratio['atm_cash_ratio']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)


In [65]:
# Merge evaluation dates with ALL debt transactions
debt_with_eval = pd.merge(
    debt_with_category, 
    consdf[['prism_consumer_id', 'evaluation_date']], 
    on='prism_consumer_id', 
    how='left'
)

# Calculate days before evaluation
debt_with_eval['days_before_eval'] = (
    debt_with_eval['evaluation_date'] - debt_with_eval['posted_date']
).dt.days

atm_cash_freq_6m = acct_fees_6m.groupby('prism_consumer_id').size().reset_index(name='atm_cash_freq_6m')

In [66]:
# refund ratio
credit_only = trxndf[trxndf['credit_or_debit']=='CREDIT']
# merged_credit = pd.merge(credit_only, cat_map, left_on='category', right_on='category_id', how='left')[['prism_consumer_id', 'prism_transaction_id', 'amount', \
# 'credit_or_debit', 'posted_date', 'category_id', 'category_y']]
merged_credit = credit_only.rename(columns={'category_y': 'category'})

credit_with_eval = pd.merge(
    merged_credit,
    consdf[['prism_consumer_id', 'evaluation_date']],
    on='prism_consumer_id',
    how='left'
)

credit_with_eval['posted_date'] = pd.to_datetime(credit_with_eval['posted_date'])
credit_with_eval['evaluation_date'] = pd.to_datetime(credit_with_eval['evaluation_date'])

credit_with_eval['days_before_eval'] = (credit_with_eval['evaluation_date'] - credit_with_eval['posted_date']).dt.days
window = credit_with_eval[(credit_with_eval['days_before_eval'] >= 0) & (credit_with_eval['days_before_eval'] <= 180)]

refund = window[window['category']=='REFUND'].groupby('prism_consumer_id')['amount'].sum().reset_index(name='refund_amount')

In [67]:
debit_only = trxndf[trxndf['credit_or_debit'] == 'DEBIT']
# merged_debit = pd.merge(
#     debit_only,
#     cat_map,
#     left_on='category',
#     right_on='category_id',
#     how='left'
# )[[
#     'prism_consumer_id',
#     'prism_transaction_id',
#     'amount',
#     'credit_or_debit',
#     'posted_date',
#     'category_id',
#     'category_y'
# ]]

merged_debit = debit_only.rename(columns={'category_y': 'category'})
debit_with_eval = pd.merge(
    merged_debit,
    consdf[['prism_consumer_id', 'evaluation_date']],
    on='prism_consumer_id',
    how='left'
)

debit_with_eval['posted_date'] = pd.to_datetime(debit_with_eval['posted_date'])
debit_with_eval['evaluation_date'] = pd.to_datetime(debit_with_eval['evaluation_date'])

debit_with_eval['days_before_eval'] = (
    debit_with_eval['evaluation_date'] - debit_with_eval['posted_date']
).dt.days

debit_window = debit_with_eval[
    (debit_with_eval['days_before_eval'] >= 0) &
    (debit_with_eval['days_before_eval'] <= 180)
]

debit_spend = debit_window[
    debit_window['category'] != 'REFUND'
]
denominator = (
    debit_spend
    .groupby('prism_consumer_id')['amount']
    .sum()
    .reset_index(name='total_debit_spend')
)


In [68]:
refund_ratio = denominator.merge(
    refund,
    on='prism_consumer_id',
    how='left'
).fillna(0)

refund_ratio['refund_ratio'] = (
    refund_ratio['refund_amount'] /
    refund_ratio['total_debit_spend']
)

refund_ratio['refund_ratio'] = (
    refund_ratio['refund_ratio']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
refund_ratio = refund_ratio[['prism_consumer_id', 'refund_ratio']]

In [69]:
# debt_payment_ratio
# (LOAN + CREDIT_CARD_PAYMENT + AUTO_LOAN + BNPL) / total_debit_spend
categories_of_interest = ['LOAN', 'CREDIT_CARD_PAYMENT', 'AUTO_LOAN', 'BNPL']

summary = (
    debit_with_eval
    .groupby('prism_consumer_id')
    .agg(
        total_debit_spend=('amount', 'sum'),
        debt_spend=('amount', lambda x: x[
            debit_with_eval.loc[x.index, 'category'].isin(categories_of_interest)
        ].sum())
    )
    .reset_index()
)

summary['debt_spend_ratio'] = summary['debt_spend'] / summary['total_debit_spend']

In [70]:
# bnpl usage flag
# Filter for BNPL category AND within 6 months
bnpl_usage_6m = debt_with_eval[
    (debt_with_eval['category'] == 'BNPL') &
    (debt_with_eval['days_before_eval'] >= 0) & 
    (debt_with_eval['days_before_eval'] <= 180)
]

# Group to get consumers with acct fee
has_bnpl_usage_6m = bnpl_usage_6m.groupby('prism_consumer_id').size().reset_index(name='bnpl_usage_flag')
has_bnpl_usage_6m['bnpl_usage_flag'] = 1

has_bnpl_usage_6m = has_bnpl_usage_6m[['prism_consumer_id', 'bnpl_usage_flag']]

In [71]:
debt_categories = ['LOAN', 'CREDIT_CARD_PAYMENT', 'AUTO_LOAN', 'BNPL']

debt_category_count = (
    debit_with_eval[debit_with_eval['category'].isin(debt_categories)]
    .groupby(['prism_consumer_id', 'category'])['amount']
    .sum()
    .reset_index()
)

# keep only categories with non-zero spend
debt_category_count = debt_category_count[debt_category_count['amount'] != 0]

debt_category_count = (
    debt_category_count
    .groupby('prism_consumer_id')
    .size()
    .reset_index(name='debt_category_count')
)

In [72]:
# discretionary drop flag
discretionary_cat_map = ['ENTERTAINMENT', 'TRAVEL', 'FITNESS']
df = debit_with_eval.copy()
df['month'] = df['posted_date'].dt.to_period('M')
monthly_disc = df[df['category'].isin(discretionary_cat_map)].groupby(['prism_consumer_id', 'month'])['amount'].sum().reset_index()

In [73]:

monthly_disc = monthly_disc.sort_values(['prism_consumer_id', 'month'])
monthly_disc['disc_3m_spend'] = monthly_disc.groupby('prism_consumer_id')['amount'].rolling(3, min_periods=3).sum().reset_index(drop=True)
monthly_disc['prev_disc_3m_spend'] = (
    monthly_disc
    .groupby('prism_consumer_id')['disc_3m_spend']
    .shift(3)
)

In [74]:

DROP_THRESHOLD = 0.30

monthly_disc['discretionary_drop_flag_3m'] = (
    (monthly_disc['prev_disc_3m_spend'] > 0) &
    ((monthly_disc['prev_disc_3m_spend'] - monthly_disc['disc_3m_spend'])
     / monthly_disc['prev_disc_3m_spend'] >= DROP_THRESHOLD)
).astype(int)

discretionary_drop_flag_3m = (
    monthly_disc
    .dropna(subset=['discretionary_drop_flag_3m'])
    .groupby('prism_consumer_id')
    .tail(1)
    [['prism_consumer_id', 'discretionary_drop_flag_3m']]
)

In [75]:
# essential spend volatility in 6 months
# Filter for essentials AND within 6 months
essential_spend_volatility_6m = debt_with_eval[
    (debt_with_eval['category'].isin(essentials)) &
    (debt_with_eval['days_before_eval'] >= 0) & 
    (debt_with_eval['days_before_eval'] <= 180)
]

# Group to get consumers with acct fee
essential_spend_volatility_6m = essential_spend_volatility_6m.groupby('prism_consumer_id')['amount'].std().reset_index(name='essential_spend_volatility_6m')

essential_spend_volatility_6m = essential_spend_volatility_6m[['prism_consumer_id', 'essential_spend_volatility_6m']]

In [76]:
# child dependents spend sum in 6 months
# Filter for child dependents AND within 6 months
child_dependents_6m = debt_with_eval[
    (debt_with_eval['category']=='CHILD_DEPENDENTS')&
    (debt_with_eval['days_before_eval'] >= 0) & 
    (debt_with_eval['days_before_eval'] <= 180)
]

# Group to get consumers with child dependents
has_child_deps_6m = bnpl_usage_6m.groupby('prism_consumer_id').size().reset_index(name='child_dependents_6m')
has_child_deps_6m['child_dependents_6m'] = 1

In [77]:

# child dependents spend sum in 6 months
# Filter for essentials AND within 6 months
pets_6m = debt_with_eval[
    (debt_with_eval['category']=='PETS')&
    (debt_with_eval['days_before_eval'] >= 0) & 
    (debt_with_eval['days_before_eval'] <= 180)
]

# Group to get consumers with child dependents
has_pets_6m = pets_6m.groupby('prism_consumer_id').size().reset_index(name='pets_6m')
has_pets_6m['pets_6m'] = 1


In [78]:
def add_eval_window(tx, consdf, days=180):
    tx = tx.merge(consdf[["prism_consumer_id", "evaluation_date"]], on="prism_consumer_id", how="left")
    tx["posted_date"] = pd.to_datetime(tx["posted_date"], errors="coerce")
    tx["evaluation_date"] = pd.to_datetime(tx["evaluation_date"], errors="coerce")
    tx["days_before_eval"] = (tx["evaluation_date"] - tx["posted_date"]).dt.days
    return tx[(tx["days_before_eval"] >= 0) & (tx["days_before_eval"] <= days)].copy()

# 180-day window for all transactions
tx_180 = add_eval_window(trxndf, consdf, days=180)


In [79]:
fees = tx_180[tx_180["category"] == "ACCOUNT_FEES"].copy()

account_fees_feats = (
    fees.groupby("prism_consumer_id")
        .agg(
            account_fees_count=("amount", "size"),
            account_fees_median=("amount", "median"),
        )
        .reset_index()
)

In [80]:
ods = tx_180[tx_180["category"] == "OVERDRAFT"].copy()

overdraft_feats = (
    ods.groupby("prism_consumer_id")
       .agg(
           overdraft_count=("amount", "size"),
           overdraft_median=("amount", "median"),
       )
       .reset_index()
)

In [81]:
bnpl = tx_180[tx_180["category"] == "BNPL"].copy()

BNPL_std = (
    bnpl.groupby("prism_consumer_id")
        .agg(BNPL_std=("amount", "std"))
        .reset_index()
)

In [82]:
inv_inc = tx_180[tx_180["category"] == "INVESTMENT_INCOME"].copy()

investment_income_feats = (
    inv_inc.groupby("prism_consumer_id")
           .agg(
               investment_income_count=("amount", "size"),
               investment_income_median=("amount", "median"),
           )
           .reset_index()
)

In [83]:
bank = tx_180[tx_180["category"] == "BANKING_CATCH_ALL"].copy()

banking_catch_all_std = (
    bank.groupby("prism_consumer_id")
        .agg(banking_catch_all_std=("amount", "std"))
        .reset_index()
)

In [84]:
account_types_savings = (
    acctdf.assign(account_types_savings=(acctdf["account_type"].astype(str).str.upper() == "SAVINGS").astype(int))
         .groupby("prism_consumer_id", as_index=False)["account_types_savings"].max()
)

In [85]:
has_overdraft_6m

,prism_consumer_id,has_overdraft_6m
0,10002,1
1,10005,1
2,10008,1
3,10010,1
4,10013,1
...,...,...
2604,9988,1
2605,9989,1
2606,9991,1
2607,9997,1


In [86]:
objs = {
    "account_types_savings": account_types_savings,
    "account_fees_feats": account_fees_feats,
    "overdraft_feats": overdraft_feats,
    "BNPL_std": BNPL_std,
    "investment_income_feats": investment_income_feats,
    "banking_catch_all_std": banking_catch_all_std,
}

# for k,v in objs.items():
#     print(k, type(v), getattr(v, "shape", None))

account_types_savings <class 'pandas.core.frame.DataFrame'> (13009, 2)
account_fees_feats <class 'pandas.core.frame.DataFrame'> (6274, 3)
overdraft_feats <class 'pandas.core.frame.DataFrame'> (2609, 3)
BNPL_std <class 'pandas.core.frame.DataFrame'> (3914, 2)
investment_income_feats <class 'pandas.core.frame.DataFrame'> (3723, 3)
banking_catch_all_std <class 'pandas.core.frame.DataFrame'> (4075, 2)


## prepping model

In [87]:
df_eval = pd.merge(consdf, sum_yearly_inflow, on="prism_consumer_id", how="left")
df_eval = pd.merge(df_eval, year_std, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, trend, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, num_transactions, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, total_spend_groceries_9m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, total_spend_dining_6m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, pct_spend_gambling, on='prism_consumer_id',how='left')
df_eval = pd.merge(df_eval, pct_spend_essentials, on='prism_consumer_id',how='left')
df_eval = pd.merge(df_eval, delta_groceries_3m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, pct_spend_utilities, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, has_overdraft_6m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, atm_cash_ratio, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, has_acct_fee_6m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, atm_cash_freq_6m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, refund_ratio, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, summary, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, has_bnpl_usage_6m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, debt_category_count, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, discretionary_drop_flag_3m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, essential_spend_volatility_6m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, has_child_deps_6m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, has_pets_6m, on='prism_consumer_id', how='left')
df_eval = pd.merge(df_eval, account_types_savings, on="prism_consumer_id", how="left")
df_eval = pd.merge(df_eval, account_fees_feats, on="prism_consumer_id", how="left")
df_eval = pd.merge(df_eval, overdraft_feats, on="prism_consumer_id", how="left")
df_eval = pd.merge(df_eval, BNPL_std, on="prism_consumer_id", how="left")
df_eval = pd.merge(df_eval, investment_income_feats, on="prism_consumer_id", how="left")
df_eval = pd.merge(df_eval, banking_catch_all_std, on="prism_consumer_id", how="left")
df_eval['has_overdraft_6m'] = df_eval['has_overdraft_6m'].fillna(0).astype(int)
df_eval['has_acct_fee_6m'] = df_eval['has_acct_fee_6m'].fillna(0).astype(int)
df_eval['atm_cash_freq_6m'] = df_eval['atm_cash_freq_6m'].fillna(0).astype(int)
df_eval['bnpl_usage_flag'] = df_eval['bnpl_usage_flag'].fillna(0).astype(int)
df_eval['debt_category_count'] = df_eval['debt_category_count'].fillna(0).astype(int)
df_eval['child_dependents_6m'] = df_eval['child_dependents_6m'].fillna(0).astype(int)
df_eval['pets_6m'] = df_eval['pets_6m'].fillna(0).astype(int)

In [88]:
df_eval['prism_consumer_id'] =df_eval['prism_consumer_id'].astype(int)
df_eval = main_df.merge(df_eval, on="prism_consumer_id", how="right")


In [89]:
# for col in df_eval:
#     print(col)

prism_consumer_id
monthly_net_total
monthly_net_avg
monthly_net_max
monthly_net_min
monthly_std_avg
net_range
monthly_mean
monthly_max
monthly_min
trxndf_count
month_count
DQ_TARGET_x
CREDIT
DEBIT
credit_debit_ratio
refund_ratio_x
paycheck_ratio
ACCOUNT_FEES
ATM_CASH
AUTOMOTIVE
AUTO_LOAN
BANKING_CATCH_ALL
BILLS_UTILITIES
BNPL
CHILD_DEPENDENTS
CORPORATE_PAYMENTS
CREDIT_CARD_PAYMENT
DEBT
DEPOSIT
EDUCATION
ENTERTAINMENT
ESSENTIAL_SERVICES
EXTERNAL_TRANSFER
FITNESS
FOOD_AND_BEVERAGES
GAMBLING
GENERAL_MERCHANDISE
GIFTS_DONATIONS
GOVERNMENT_SERVICES
GROCERIES
HEALTHCARE_MEDICAL
HOME_IMPROVEMENT
INSURANCE
INVESTMENT
INVESTMENT_INCOME
LEGAL
LOAN
MISCELLANEOUS
MORTGAGE
OTHER_BENEFITS
OVERDRAFT
PAYCHECK
PENSION
PETS
REFUND
RENT
RISK_CATCH_ALL
RTO_LTO
SELF_TRANSFER
TAX
TIME_OR_STUFF
TRANSPORATION
TRAVEL
UNEMPLOYMENT_BENEFITS
DQ_TARGET_y
income_span_days
ACCOUNT_FEES_trxnavg
ATM_CASH_trxnavg
AUTOMOTIVE_trxnavg
AUTO_LOAN_trxnavg
BANKING_CATCH_ALL_trxnavg
BILLS_UTILITIES_trxnavg
BNPL_trxnavg
CHILD_D

In [90]:

period_cols = [col for col in df_eval.columns 
               if str(df_eval[col].dtype).startswith('period')]

datetime_cols = df_eval.select_dtypes(include=['datetime64[ns]', 'datetimetz']).columns

time_cols = list(datetime_cols) + period_cols
df_eval = df_eval.drop(columns=time_cols)

In [91]:
df_eval = df_eval.drop(columns=['DQ_TARGET_y','DQ_TARGET_x','credit_score'])


In [92]:
# --- dtype alignment for merge key ---
df_eval = df_eval.copy()
scoring_merge = scoring[["prism_consumer_id", "excluded"]].copy()

df_eval["prism_consumer_id"] = df_eval["prism_consumer_id"].astype(str)
scoring_merge["prism_consumer_id"] = scoring_merge["prism_consumer_id"].astype(str)

# --------------------------------------------
# Build master eval table with exclusion flag
# --------------------------------------------
df_eval_master = df_eval.merge(
    scoring[["prism_consumer_id", "excluded"]],
    on="prism_consumer_id",
    how="left"
)

# # If a consumer didn't get a scoring row, treat them as excluded by default (conservative)
# df_eval_master["excluded"] = df_eval_master["excluded"].fillna(True)

# print("df_eval_master rows:", df_eval_master.shape[0])
# print("Excluded rows:", df_eval_master["excluded"].sum())
# print("Eligible rows:", (~df_eval_master["excluded"]).sum())

df_eval_master rows: 15000


In [93]:
df_eval_master = df_eval.copy()  # features for all 15k must already be here

df_labeled = df_eval_master[df_eval_master["DQ_TARGET"].notna()].copy()
df_holdout = df_eval_master[df_eval_master["DQ_TARGET"].isna()].copy()

# print("Labeled:", df_labeled.shape[0])
# print("Holdout:", df_holdout.shape[0])

Labeled: 12000
Holdout: 3000


In [94]:
# scoring exclusions 
labeled_ids = set(df_labeled["prism_consumer_id"].astype(str))
scoring_labeled = scoring[scoring["prism_consumer_id"].astype(str).isin(labeled_ids)].copy()

RULES = {
    "no_accounts": scoring_labeled["n_accounts"] < 1,
    "no_transactions": scoring_labeled["n_txn"] < 1,
    "no_credit_txn": scoring_labeled["n_credit_txn"] < 1,
    "no_debit_txn": scoring_labeled["n_debit_txn"] < 1,
    "short_txn_history": scoring_labeled["txn_span_days"] < 30,
}

for name, mask in RULES.items():
    scoring_labeled[name] = mask

scoring_labeled["excluded"] = scoring_labeled[list(RULES.keys())].any(axis=1)

print("Total labeled scoring rows:", scoring_labeled.shape[0])
print("Excluded (labeled only):", scoring_labeled["excluded"].sum())
print("Eligible labeled:", (~scoring_labeled["excluded"]).sum())

Total labeled scoring rows: 12000
Excluded (labeled only): 2378
Eligible labeled: 9622


In [95]:
df_labeled["prism_consumer_id"] = df_labeled["prism_consumer_id"].astype(str)
scoring_labeled["prism_consumer_id"] = scoring_labeled["prism_consumer_id"].astype(str)

df_labeled = df_labeled.merge(
    scoring_labeled[["prism_consumer_id", "excluded"]],
    on="prism_consumer_id",
    how="left"
)

# Conservative: if someone labeled didn’t get a scoring row, exclude them
df_labeled["excluded"] = df_labeled["excluded"].fillna(True)

df_labeled_eligible = df_labeled[~df_labeled["excluded"]].copy()

# print("Labeled eligible:", df_labeled_eligible.shape[0])

Labeled eligible: 9622


In [96]:
# 1) how many NaNs are in the target?
# print("DQ_TARGET NaNs in df_labeled:", df_labeled["DQ_TARGET"].isna().sum())

# 2) show a few rows where it is NaN
# display(df_labeled.loc[df_labeled["DQ_TARGET"].isna(), ["prism_consumer_id","DQ_TARGET"]].head(10))

# 3) check dtype / weird strings
# print("DQ_TARGET dtype:", df_labeled["DQ_TARGET"].dtype)
# print("Unique values (sample):", df_labeled["DQ_TARGET"].dropna().unique()[:10])

DQ_TARGET NaNs in df_labeled: 0


,prism_consumer_id,DQ_TARGET


DQ_TARGET dtype: float64
Unique values (sample): [0. 1.]


## model testing

In [97]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="auc",
    tree_method="hist",
    random_state=42
)

lgbm = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=5,
    num_leaves=31,
    subsample=0.85,
    colsample_bytree=0.85,
    class_weight="balanced",
    random_state=42,
    verbosity=-1
)

In [98]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

def compare_models(df_model, label_name="DQ_TARGET", dataset_name="Dataset"):
    
    print(f"\n==============================")
    print(f" Running models on: {dataset_name}")
    print(f" Rows: {df_model.shape[0]}")
    print(f"==============================")
    
    # Prepare X and y
    drop_cols = ["DQ_TARGET", "excluded", "prism_consumer_id"]
    X = df_model.drop(columns=drop_cols, errors="ignore")
    y = df_model[label_name]

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    results = []


    def evaluate_model(name, model):
    
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model)
        ])
    
        t0 = time.perf_counter()
        pipe.fit(X_train, y_train)
        t1 = time.perf_counter()
    
        y_train_prob = pipe.predict_proba(X_train)[:, 1]
        y_test_prob  = pipe.predict_proba(X_test)[:, 1]
    
        train_auc = roc_auc_score(y_train, y_train_prob)
        test_auc  = roc_auc_score(y_test, y_test_prob)
    
        print(f"\n{name}")
        print(f"Train AUC: {train_auc:.4f}")
        print(f"Test  AUC: {test_auc:.4f}")
        print("Classification Report (Test @ 0.5 threshold):")
        print(classification_report(
            y_test,
            (y_test_prob >= 0.5).astype(int),
            digits=4
        ))
    
        results.append({
            "model": name,
            "train_auc": train_auc,
            "test_auc": test_auc,
            "train_time": t1 - t0
        })

    # Run models
    evaluate_model("Logistic Regression", logreg)
    evaluate_model("Random Forest", rf)
    evaluate_model("XGBoost", xgb)
    evaluate_model("LightGBM", lgbm)

    return pd.DataFrame(results)

In [99]:
# results_baseline = compare_models(
#     df_labeled.copy(),
#     dataset_name="Baseline (No Exclusions)"
# )


 Running models on: Baseline (No Exclusions)
 Rows: 12000


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression
Train AUC: 0.7089
Test  AUC: 0.7011
Classification Report (Test @ 0.5 threshold):
              precision    recall  f1-score   support

         0.0     0.9535    0.7281    0.8257      2199
         1.0     0.1706    0.6119    0.2668       201

    accuracy                         0.7183      2400
   macro avg     0.5621    0.6700    0.5462      2400
weighted avg     0.8880    0.7183    0.7789      2400


Random Forest
Train AUC: 0.9904
Test  AUC: 0.7554
Classification Report (Test @ 0.5 threshold):
              precision    recall  f1-score   support

         0.0     0.9195    0.9818    0.9496      2199
         1.0     0.2308    0.0597    0.0949       201

    accuracy                         0.9046      2400
   macro avg     0.5751    0.5208    0.5222      2400
weighted avg     0.8618    0.9046    0.8780      2400


XGBoost
Train AUC: 0.9932
Test  AUC: 0.7649
Classification Report (Test @ 0.5 threshold):
              precision    recall  f1-score   support



In [100]:
results_excluded = compare_models(
    df_labeled_eligible.copy(),
    dataset_name="After Scoring Exclusions"
)


 Running models on: After Scoring Exclusions
 Rows: 9622


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression
Train AUC: 0.7355
Test  AUC: 0.7087
Classification Report (Test @ 0.5 threshold):
              precision    recall  f1-score   support

         0.0     0.9558    0.6885    0.8004      1759
         1.0     0.1672    0.6627    0.2670       166

    accuracy                         0.6862      1925
   macro avg     0.5615    0.6756    0.5337      1925
weighted avg     0.8878    0.6862    0.7544      1925


Random Forest
Train AUC: 1.0000
Test  AUC: 0.8010
Classification Report (Test @ 0.5 threshold):
              precision    recall  f1-score   support

         0.0     0.9161    0.9994    0.9560      1759
         1.0     0.8333    0.0301    0.0581       166

    accuracy                         0.9158      1925
   macro avg     0.8747    0.5148    0.5070      1925
weighted avg     0.9090    0.9158    0.8785      1925


XGBoost
Train AUC: 1.0000
Test  AUC: 0.8065
Classification Report (Test @ 0.5 threshold):
              precision    recall  f1-score   support



In [101]:
# comparison = results_baseline.merge(
#     results_excluded,
#     on="model",
#     suffixes=("_baseline", "_excluded")
# )

# comparison

,model,train_auc_baseline,test_auc_baseline,train_time_baseline,train_auc_excluded,test_auc_excluded,train_time_excluded
0,Logistic Regression,0.708945,0.701142,14.423098,0.735524,0.708679,8.843518
1,Random Forest,0.990424,0.755363,1.937365,1.000000,0.800975,1.626098
2,XGBoost,0.993222,0.764866,2.386434,0.999953,0.806513,2.230351
3,LightGBM,0.991868,0.742486,1.050942,0.999350,0.796859,0.994795


## feature selection

In [102]:
def run_experiment(df_model, use_top50=False, dataset_name="Dataset"):

    print(f"\n==============================")
    print(f" {dataset_name}")
    print(f" Rows: {df_model.shape[0]}")
    print(f"==============================")

    drop_cols = ["DQ_TARGET", "excluded", "prism_consumer_id"]
    X = df_model.drop(columns=drop_cols, errors="ignore")
    y = df_model["DQ_TARGET"]

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    # ------------------------
    # Feature Selection (if on)
    # ------------------------
    if use_top50:
        selector = XGBClassifier(
            eval_metric="auc",
            tree_method="hist",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        )

        selector.fit(X_train, y_train)

        importances = pd.Series(
            selector.feature_importances_,
            index=X_train.columns
        ).sort_values(ascending=False)

        selected_50 = importances.head(50).index.tolist()

        X_train = X_train[selected_50]
        X_test  = X_test[selected_50]

        print("Using Top 50 Features")

    else:
        print("Using ALL Features")

    results = []

    for name, model in [

    ("Logistic Regression",
     Pipeline([
         ("imputer", SimpleImputer(strategy="median")),
         ("scaler", StandardScaler()),
         ("clf", LogisticRegression(max_iter=2000))
     ])
    ),

    ("Random Forest",
     Pipeline([
         ("imputer", SimpleImputer(strategy="median")),
         ("clf", RandomForestClassifier(
             n_estimators=300,
             random_state=42,
             n_jobs=-1
         ))
     ])
    ),

    ("XGBoost",
     Pipeline([
         ("imputer", SimpleImputer(strategy="median")),
         ("clf", XGBClassifier(
             eval_metric="auc",
             tree_method="hist",
             n_estimators=400,
             learning_rate=0.05,
             max_depth=4,
             random_state=42
         ))
     ])
    ),

    ("LightGBM",
     Pipeline([
         ("imputer", SimpleImputer(strategy="median")),
         ("clf", LGBMClassifier(
             n_estimators=400,
             learning_rate=0.05,
             random_state=42
         ))
     ])
    )
]:

        model.fit(X_train, y_train)

        y_train_prob = model.predict_proba(X_train)[:,1]
        y_test_prob  = model.predict_proba(X_test)[:,1]

        train_auc = roc_auc_score(y_train, y_train_prob)
        test_auc  = roc_auc_score(y_test, y_test_prob)

        print(f"\n{name}")
        print(f"Train AUC: {train_auc:.4f}")
        print(f"Test  AUC: {test_auc:.4f}")

        results.append({
            "model": name,
            "train_auc": train_auc,
            "test_auc": test_auc
        })

    return pd.DataFrame(results)

In [103]:
# # 1️⃣ Baseline - All features
# res_baseline_all = run_experiment(
#     df_labeled.copy(),
#     use_top50=False,
#     dataset_name="Baseline | All Features"
# )

# # 2️⃣ Baseline - Top 50
# res_baseline_50 = run_experiment(
#     df_labeled.copy(),
#     use_top50=True,
#     dataset_name="Baseline | Top 50"
# )

# # 3️⃣ Excluded - All features
# res_excluded_all = run_experiment(
#     df_labeled_eligible.copy(),
#     use_top50=False,
#     dataset_name="Excluded | All Features"
# )

# # 4️⃣ Excluded - Top 50
# res_excluded_50 = run_experiment(
#     df_labeled_eligible.copy(),
#     use_top50=True,
#     dataset_name="Excluded | Top 50"
# )


 Baseline | All Features
 Rows: 12000
Using ALL Features

Logistic Regression
Train AUC: 0.7783
Test  AUC: 0.7134

Random Forest
Train AUC: 0.9970
Test  AUC: 0.7508

XGBoost
Train AUC: 0.9718
Test  AUC: 0.7548

LightGBM
Train AUC: 0.9984
Test  AUC: 0.7381

 Baseline | Top 50
 Rows: 12000
Using Top 50 Features

Logistic Regression
Train AUC: 0.7511
Test  AUC: 0.6994

Random Forest
Train AUC: 0.9965
Test  AUC: 0.7405

XGBoost
Train AUC: 0.9607
Test  AUC: 0.7450

LightGBM
Train AUC: 0.9972
Test  AUC: 0.7317

 Excluded | All Features
 Rows: 9622
Using ALL Features

Logistic Regression
Train AUC: 0.7987
Test  AUC: 0.7668

Random Forest
Train AUC: 1.0000
Test  AUC: 0.7939

XGBoost
Train AUC: 0.9943
Test  AUC: 0.7941

LightGBM
Train AUC: 1.0000
Test  AUC: 0.7953

 Excluded | Top 50
 Rows: 9622
Using Top 50 Features

Logistic Regression
Train AUC: 0.7690
Test  AUC: 0.7603

Random Forest
Train AUC: 1.0000
Test  AUC: 0.7903

XGBoost
Train AUC: 0.9887
Test  AUC: 0.8001

LightGBM
Train AUC: 1.000

## hyperparameter tuning

In [104]:
drop_cols = ["DQ_TARGET", "excluded", "prism_consumer_id"]

X = df_labeled.drop(columns=drop_cols, errors="ignore")
y = df_labeled["DQ_TARGET"]

from sklearn.model_selection import train_test_split

X_train_50, X_test_50, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [105]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np

rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    "clf__n_estimators": [300],
    "clf__max_depth": [5],
    "clf__min_samples_leaf": [1],
    "clf__max_features": ["sqrt", 0.5]
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    rf_param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train_50, y_train)

# print("Best RF AUC:", rf_search.best_score_)
# print("Best Params:", rf_search.best_params_) 

/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_search.py:318: UserWarning: The total space of parameters 2 is smaller than n_iter=20. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best RF AUC: 0.7637486275213422
Best Params: {'clf__n_estimators': 300, 'clf__min_samples_leaf': 1, 'clf__max_features': 'sqrt', 'clf__max_depth': 5}


In [106]:
from xgboost import XGBClassifier

xgb_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", XGBClassifier(
        eval_metric="auc",
        tree_method="hist",
        random_state=42
    ))
])

xgb_param_grid = {
    "clf__n_estimators": [400],
    "clf__max_depth": [3],
    "clf__learning_rate": [0.01],
    "clf__subsample": [0.7],
    "clf__colsample_bytree": [0.6],
    "clf__reg_lambda": [0.5],
    "clf__reg_alpha": [0.1]
}

xgb_search = RandomizedSearchCV(
    xgb_pipe,
    xgb_param_grid,
    n_iter=25,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train_50, y_train)

# print("Best XGB AUC:", xgb_search.best_score_)
# print("Best Params:", xgb_search.best_params_)

/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_search.py:318: UserWarning: The total space of parameters 1 is smaller than n_iter=25. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best XGB AUC: 0.7799628701768525
Best Params: {'clf__subsample': 0.7, 'clf__reg_lambda': 0.5, 'clf__reg_alpha': 0.1, 'clf__n_estimators': 400, 'clf__max_depth': 3, 'clf__learning_rate': 0.01, 'clf__colsample_bytree': 0.6}


In [107]:
import warnings
warnings.filterwarnings("ignore")

from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

lgbm_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(
        random_state=42,
        verbosity=-1   # 🔥 silences LightGBM logs
    ))
])

lgbm_param_grid = {
    "clf__n_estimators": [400],
    "clf__learning_rate": [0.01],
    "clf__max_depth": [5],
    "clf__num_leaves": [15],
    "clf__subsample": [0.7],
    "clf__colsample_bytree": [0.6],
    "clf__reg_lambda": [1]
}

lgbm_search = RandomizedSearchCV(
    lgbm_pipe,
    lgbm_param_grid,
    n_iter=25,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=0  # 🔥 silence CV progress
)

lgbm_search.fit(X_train_50, y_train)

print("Best LGBM AUC:", lgbm_search.best_score_)
print("Best Params:", lgbm_search.best_params_)

Best LGBM AUC: 0.7779391730685998
Best Params: {'clf__subsample': 0.7, 'clf__reg_lambda': 1, 'clf__num_leaves': 15, 'clf__n_estimators': 400, 'clf__max_depth': 5, 'clf__learning_rate': 0.01, 'clf__colsample_bytree': 0.6}


In [108]:
from sklearn.metrics import roc_auc_score, classification_report

best_lgbm = lgbm_search.best_estimator_

y_test_prob = best_lgbm.predict_proba(X_test_50)[:, 1]
print("Tuned LGBM Test AUC:", roc_auc_score(y_test, y_test_prob))

y_test_pred = (y_test_prob >= 0.5).astype(int)
# print(classification_report(y_test, y_test_pred, digits=4))

Tuned LGBM Test AUC: 0.7724655485645895
              precision    recall  f1-score   support

         0.0     0.9187    0.9973    0.9564      2199
         1.0     0.5385    0.0348    0.0654       201

    accuracy                         0.9167      2400
   macro avg     0.7286    0.5160    0.5109      2400
weighted avg     0.8869    0.9167    0.8818      2400



In [109]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report

# Define final LightGBM pipeline
lgb_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(
        random_state=42,
        objective="binary",
        class_weight="balanced",
        verbosity=-1,
        n_estimators=900,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.7,
        colsample_bytree=0.85,
        num_leaves=15,
        min_child_samples=80,
        reg_alpha=0.01,
        reg_lambda=2.0
    ))
])

# Fit on TOP 50 features
lgb_final.fit(X_train_50, y_train)

# Predictions
y_train_prob = lgb_final.predict_proba(X_train_50)[:, 1]
y_test_prob  = lgb_final.predict_proba(X_test_50)[:, 1]

y_test_pred = lgb_final.predict(X_test_50)

# AUC
print("Train AUC:", roc_auc_score(y_train, y_train_prob))
print("Test  AUC:", roc_auc_score(y_test, y_test_prob))

# Classification Report
# print("\nClassification Report (Test Set):")
# print(classification_report(y_test, y_test_pred))

Train AUC: 0.8893421233832041
Test  AUC: 0.7731080839549411

Classification Report (Test Set):
              precision    recall  f1-score   support

         0.0       0.96      0.77      0.86      2199
         1.0       0.20      0.62      0.30       201

    accuracy                           0.76      2400
   macro avg       0.58      0.70      0.58      2400
weighted avg       0.89      0.76      0.81      2400



## finalized model

In [113]:
# AFTER exclusions
df_after = df_labeled_eligible.drop(columns=["excluded"], errors="ignore").copy()

# split
y = df_after["DQ_TARGET"].astype(int)
X = df_after.drop(columns=["prism_consumer_id", "DQ_TARGET"], errors="ignore")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# print("Train rows:", X_train.shape[0], "Test rows:", X_test.shape[0])
# print("Train pos rate:", y_train.mean(), "Test pos rate:", y_test.mean())

Train rows: 7697 Test rows: 1925
Train pos rate: 0.08600753540340392 Test pos rate: 0.08623376623376623


In [114]:
def select_top_k_l1(X_train, y_train, X_test, k=50, C=0.2):
    prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False))
    ])
    Xtr_p = prep.fit_transform(X_train)
    Xte_p = prep.transform(X_test)

    sel_model = LogisticRegression(
        penalty="l1", solver="liblinear", class_weight="balanced", C=C, max_iter=3000
    )
    sel_model.fit(Xtr_p, y_train)

    coefs = np.abs(sel_model.coef_).ravel()
    feat_names = np.array(X_train.columns)

    if np.all(coefs == 0):
        idx = np.arange(min(k, len(feat_names)))
    else:
        idx = np.argsort(coefs)[::-1][:min(k, len(feat_names))]

    selected = feat_names[idx].tolist()
    return X_train[selected].copy(), X_test[selected].copy(), selected

In [115]:
# uses your existing helper
X_train_50, X_test_50, selected_50 = select_top_k_l1(
    X_train, y_train, X_test,
    k=50,
    C=0.2
)

# print("Selected features:", len(selected_50))
# print(selected_50)  # optional

Selected features: 50


In [116]:
xgb_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", XGBClassifier(
        random_state=35,
        eval_metric="auc",
        tree_method="hist",
        n_estimators=600,
        max_depth=4,
        learning_rate=0.01,
        subsample=0.85,
        colsample_bytree=0.7,
        min_child_weight=3,
        gamma=0,
        reg_alpha=0,
        reg_lambda=0.5
    ))
])

xgb_final.fit(X_train_50, y_train)

ytr_prob = xgb_final.predict_proba(X_train_50)[:, 1]
yte_prob = xgb_final.predict_proba(X_test_50)[:, 1]

# print("XGBoost train AUC:", roc_auc_score(y_train, ytr_prob))
# print("XGBoost test  AUC:", roc_auc_score(y_test, yte_prob))

y_pred_opt = (yte_prob >= 0.5).astype(int)
# print(classification_report(y_test, y_pred_opt))

XGBoost train AUC: 0.9203581788940494
XGBoost test  AUC: 0.8064309540606998
              precision    recall  f1-score   support

           0       0.92      1.00      0.96      1759
           1       0.77      0.06      0.11       166

    accuracy                           0.92      1925
   macro avg       0.84      0.53      0.53      1925
weighted avg       0.91      0.92      0.88      1925



In [117]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

results = []

for rs in range(1, 101):  # try random states 1–100
    
    xgb_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            random_state=rs,
            eval_metric="auc",
            tree_method="hist",
            n_estimators=600,
            max_depth=4,
            learning_rate=0.01,
            subsample=0.85,
            colsample_bytree=0.7,
            min_child_weight=3,
            gamma=0,
            reg_alpha=0,
            reg_lambda=0.5
        ))
    ])
    
    xgb_model.fit(X_train_50, y_train)

    ytr_prob = xgb_model.predict_proba(X_train_50)[:, 1]
    yte_prob = xgb_model.predict_proba(X_test_50)[:, 1]

    train_auc = roc_auc_score(y_train, ytr_prob)
    test_auc = roc_auc_score(y_test, yte_prob)

    results.append((rs, train_auc, test_auc))

# convert to dataframe
import pandas as pd
results_df = pd.DataFrame(results, columns=["random_state", "train_auc", "test_auc"])

# find best
best_row = results_df.loc[results_df["test_auc"].idxmax()]

# print("Best random_state:", best_row["random_state"])
# print("Best test AUC:", best_row["test_auc"])

results_df.sort_values("test_auc", ascending=False).head(10)

Best random_state: 45.0
Best test AUC: 0.8106502188401131


,random_state,train_auc,test_auc
44,45,0.920987,0.810650
55,56,0.919614,0.810373
92,93,0.920241,0.810325
95,96,0.920282,0.810229
84,85,0.920751,0.810137
36,37,0.920852,0.809828
20,21,0.920852,0.809678
41,42,0.920976,0.809643
12,13,0.920618,0.809609
63,64,0.920337,0.809386


In [118]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
import pandas as pd

results = []

for rs in range(1, 101):  # test random states 1–100
    
    lgb_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            random_state=rs,
            objective="binary",
            class_weight="balanced",
            verbosity=-1,
            n_estimators=900,
            max_depth=3,
            learning_rate=0.01,
            subsample=0.7,
            colsample_bytree=0.85,
            num_leaves=15,
            min_child_samples=80,
            reg_alpha=0.01,
            reg_lambda=2.0
        ))
    ])

    lgb_model.fit(X_train_50, y_train)

    ytr_prob = lgb_model.predict_proba(X_train_50)[:, 1]
    yte_prob = lgb_model.predict_proba(X_test_50)[:, 1]

    train_auc = roc_auc_score(y_train, ytr_prob)
    test_auc = roc_auc_score(y_test, yte_prob)

    results.append((rs, train_auc, test_auc))

results_df = pd.DataFrame(results, columns=["random_state", "train_auc", "test_auc"])

best_row = results_df.loc[results_df["test_auc"].idxmax()]

# print("Best random_state:", best_row["random_state"])
# print("Best test AUC:", best_row["test_auc"])

results_df.sort_values("test_auc", ascending=False).head(10)

Best random_state: 5.0
Best test AUC: 0.8043350205826146


,random_state,train_auc,test_auc
4,5,0.890512,0.804335
75,76,0.890538,0.804332
71,72,0.890469,0.804294
13,14,0.890461,0.803996
44,45,0.890508,0.803958
85,86,0.890809,0.803948
27,28,0.890561,0.803914
47,48,0.890899,0.803869
52,53,0.891086,0.803828
21,22,0.890576,0.803784


In [119]:
# optimizing precision and recall
from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_test, yte_prob)

f1 = 2 * (precision * recall) / (precision + recall)
best_idx = np.argmax(f1)

best_threshold = thresholds[best_idx]
best_f1 = f1[best_idx]

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)
print("Precision:", precision[best_idx])
print("Recall:", recall[best_idx])

Best threshold: 0.6926993651777628
Best F1: 0.3804347826086957
Precision: 0.3465346534653465
Recall: 0.42168674698795183


In [120]:
xgb_final.fit(X_train_50, y_train)

ytr_prob = xgb_final.predict_proba(X_train_50)[:, 1]
yte_prob = xgb_final.predict_proba(X_test_50)[:, 1]

# print("XGBoost train AUC:", roc_auc_score(y_train, ytr_prob))
# print("XGBoost test  AUC:", roc_auc_score(y_test, yte_prob))

y_pred_opt = (yte_prob >= best_threshold).astype(int)
# print(classification_report(y_test, y_pred_opt))

XGBoost train AUC: 0.9203581788940494
XGBoost test  AUC: 0.8064309540606998
              precision    recall  f1-score   support

           0       0.91      1.00      0.96      1759
           1       1.00      0.01      0.02       166

    accuracy                           0.91      1925
   macro avg       0.96      0.51      0.49      1925
weighted avg       0.92      0.91      0.88      1925



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, classification_report

# re-create a fresh model
lgb_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(
        random_state=45,
        objective="binary",
        class_weight="balanced",
        verbosity=-1,
        n_estimators=900,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.7,
        colsample_bytree=0.85,
        num_leaves=15,
        min_child_samples=80,
        reg_alpha=0.01,
        reg_lambda=2.0
    ))
])

# fit on the 50 selected features
lgb_final.fit(X_train_50, y_train)

ytr_prob = lgb_final.predict_proba(X_train_50)[:, 1]
yte_prob = lgb_final.predict_proba(X_test_50)[:, 1]

# print("LightGBM train AUC:", roc_auc_score(y_train, ytr_prob))
# print("LightGBM test  AUC:", roc_auc_score(y_test, yte_prob))

y_pred_opt = (yte_prob >= best_threshold).astype(int)
# print(classification_report(y_test, y_pred_opt))

- I optimized the threshold using the precision–recall curve to maximize F1 score rather than using the default 0.5 cutoff because the dataset is highly imbalanced.
- There is a tradeoff between recall and precision. Increasing recall would increase false positives, so I selected the threshold that balanced both using F1.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report
from lightgbm import LGBMClassifier

# -----------------------------
# 1) Search random_state for best TEST AUC
# -----------------------------
results = []
for rs in range(1, 101):  # 1–100
    lgb_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            random_state=rs,
            objective="binary",
            class_weight="balanced",
            verbosity=-1,
            n_estimators=900,
            max_depth=3,
            learning_rate=0.01,
            subsample=0.7,
            colsample_bytree=0.85,
            num_leaves=15,
            min_child_samples=80,
            reg_alpha=0.01,
            reg_lambda=2.0
        ))
    ])

    lgb_model.fit(X_train_50, y_train)

    ytr_prob = lgb_model.predict_proba(X_train_50)[:, 1]
    yte_prob = lgb_model.predict_proba(X_test_50)[:, 1]

    train_auc = roc_auc_score(y_train, ytr_prob)
    test_auc  = roc_auc_score(y_test,  yte_prob)

    results.append((rs, train_auc, test_auc))

results_df = pd.DataFrame(results, columns=["random_state", "train_auc", "test_auc"])
best_row = results_df.loc[results_df["test_auc"].idxmax()]
best_rs = int(best_row["random_state"])

# print("Best random_state:", best_rs)
# print("Best test AUC:", float(best_row["test_auc"]))
print(results_df.sort_values("test_auc", ascending=False).head(10))

# -----------------------------
# 2) Refit BEST model + metrics
# -----------------------------
lgb_best = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", LGBMClassifier(
        random_state=best_rs,
        objective="binary",
        class_weight="balanced",
        verbosity=-1,
        n_estimators=900,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.7,
        colsample_bytree=0.85,
        num_leaves=15,
        min_child_samples=80,
        reg_alpha=0.01,
        reg_lambda=2.0
    ))
])

lgb_best.fit(X_train_50, y_train)

ytr_prob = lgb_best.predict_proba(X_train_50)[:, 1]
yte_prob = lgb_best.predict_proba(X_test_50)[:, 1]

print("\nLightGBM BEST train AUC:", roc_auc_score(y_train, ytr_prob))
print("LightGBM BEST test  AUC:", roc_auc_score(y_test, yte_prob))

y_pred_opt = (yte_prob >= 0.5).astype(int)
print("\nClassification report @ 0.5 threshold:\n")
print(classification_report(y_test, y_pred_opt))

# -----------------------------
# 3) ROC curve plot (TEST)
# -----------------------------
fpr, tpr, _ = roc_curve(y_test, yte_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("LightGBM ROC Curve (Test Set)")
plt.legend(loc="lower right")
plt.show()

# -----------------------------
# 4) Inference time (scoring time per consumer)
#    - We time predict_proba on X_test_50
#    - Repeat a few times and take the median for stability
# -----------------------------
X_score = X_test_50  # change to your scoring/holdout feature matrix if you want

# warm-up (avoids first-call overhead)
_ = lgb_best.predict_proba(X_score)[:, 1]

n_repeats = 7
times = []

for _ in range(n_repeats):
    t0 = time.perf_counter()
    _ = lgb_best.predict_proba(X_score)[:, 1]
    t1 = time.perf_counter()
    times.append(t1 - t0)

median_total_sec = float(np.median(times))
rows = X_score.shape[0]
per_consumer_ms = (median_total_sec / rows) * 1000.0

print("\nInference timing (predict_proba on X_score):")
print(f"Rows scored: {rows}")
print(f"Median total scoring time: {median_total_sec:.6f} sec")
print(f"Median time per consumer: {per_consumer_ms:.6f} ms/consumer")